# 00 患者匹配、影像检查与四层缓存

鉴于原始数据集有20多个G,不利于整个上传,先在本机处理一部分数据
这本 Notebook 在原始数据所在的机器上完成患者匹配、影像核查、四层缓存和时间分组，再导出分类训练需要上传的数据包。

下面定位配置文件所在目录。数据生成步骤各有开关，按顺序开启需要执行的步骤。


In [3]:
from pathlib import Path

HERE = Path.cwd().resolve()
CANDIDATES = [HERE, HERE / "1117", HERE.parent / "1117"]
NB_ROOT = next((p for p in CANDIDATES if (p / "config.json").is_file()), None)
if NB_ROOT is None:
    raise FileNotFoundError("请从保存config.json的Notebook目录打开；不需要外层code目录。")


## 实验配置与文件记录

这一组函数负责管理实验文件。`load_config` 读取 `config.json`，`paths` 整理缓存、模型和报告的保存位置；后面的单元通过 `cfg` 读取参数，通过 `p` 查找文件。

`read_cohort` 读取患者清单，`development` 从中取出开发队列。保存结果时，`record` 会同时记录配置和代码版本，便于回查某次结果是怎样得到的。`fresh` 用来检查目标文件是否已存在，重新开展一组实验时可以换一个 `run_id`。


In [ ]:
"""Configuration, provenance and stage boundaries; standard library only."""
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path

SEED = 20260922
CLINICAL = ["age", "education", "nihss", "hcy", "ua"]
SEQUENCES = ["t1", "t2", "dwi"]
IMAGE_FILES = ["t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz"]


def load_config(path: str | Path) -> dict:
    path = Path(path).resolve()
    cfg = json.loads(path.read_text(encoding="utf-8"))
    if not re.fullmatch(r"[A-Za-z0-9_-]+", cfg["run_id"]):
        raise ValueError("run_id 只使用字母、数字、下划线或短横线。")
    cfg["_root"] = path.parent
    for key in ("excel_path", "patients_root", "existing_cache", "outputs_root"):
        if cfg.get(key):
            p = Path(cfg[key]).expanduser()
            cfg[key] = p if p.is_absolute() else path.parent / p
    cfg["_run"] = cfg["outputs_root"] / cfg["run_id"]
    if cfg.get("data_access") == "classification_cache_only":
        verify_classification_bundle(cfg)
    return cfg


def paths(cfg: dict) -> dict[str, Path]:
    root = cfg["_run"]
    return {
        "run": root, "data": root / "data", "cache": Path(cfg.get("existing_cache") or root / "data" / "resnet50_manual_cache.npz"),
        "adapter": root / "data" / "clinical_adapter.xlsx", "cohort": root / "data" / "cohort.csv",
        "temporal": root / "temporal", "full_cv": root / "full_cv", "single": root / "single_sequence",
        "radiomics": root / "radiomics", "explain": root / "explain", "segmentation": root / "segmentation",
        "reports": root / "reports",
    }


def dump_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str, allow_nan=False), encoding="utf-8")


def fresh(*targets: Path) -> None:
    exists = [str(p) for p in targets if p.exists()]
    if exists:
        raise FileExistsError("产物已存在，请加载已有结果，或更换 run_id：" + ", ".join(exists))


def digest(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def source_snapshot(cfg: dict) -> dict:
    """Hash the literal definition cells, independently of outputs and execution counts."""
    result = {}
    for path in sorted(cfg["_root"].glob("*.ipynb")):
        notebook = json.loads(path.read_text(encoding="utf-8"))
        definitions = ["".join(cell["source"]) for cell in notebook["cells"]
                       if cell["cell_type"] == "code" and cell.get("metadata", {}).get("embedded_definition")]
        result[path.name] = hashlib.sha256("\n\n".join(definitions).encode("utf-8")).hexdigest()
    return result


def record(cfg: dict, stage: str, extra: dict | None = None) -> None:
    obj = {"stage": stage, "config": {k: v for k, v in cfg.items() if not k.startswith("_")},
           "notebook_definitions_sha256": source_snapshot(cfg),
           "protocol": "original_resnet_integration_no_architecture_optimization", **(extra or {})}
    dump_json(paths(cfg)["run"] / "provenance" / f"{stage}.json", obj)


def read_cohort(cfg: dict):
    import pandas as pd
    p = paths(cfg)["cohort"]
    if not p.exists():
        raise FileNotFoundError("先运行00的数据准备与病例清单单元。")
    table = pd.read_csv(p, dtype={"mr": str})
    if not table["mr"].is_unique or not set(table["split"]) <= {"development", "validation"}:
        raise ValueError("病例清单编号重复或队列标记无效。")
    return table


def development(cfg: dict):
    return read_cohort(cfg).query("split == 'development'").reset_index(drop=True)


def require_input(cfg: dict, key: str) -> Path:
    value = cfg.get(key)
    if not value or not Path(value).exists():
        raise FileNotFoundError(f"请在 config.json 配置有效的 {key}。")
    return Path(value)


def verify_classification_bundle(cfg: dict) -> None:
    """Verify portable data without accessing the original image directories."""
    root = cfg["_root"].resolve()
    manifest_path = root / "classification_bundle.json"
    if not manifest_path.is_file():
        raise FileNotFoundError("分类数据包缺少 classification_bundle.json，请上传完整目录。")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("kind") != "classification_cache_only" or manifest["run_id"] != cfg["run_id"]:
        raise ValueError("分类数据包与当前 run_id 不一致。")
    p = paths(cfg)
    for key in ("cache", "cohort"):
        expected = (root / manifest[key]).resolve()
        expected.relative_to(root)
        if p[key].resolve() != expected or manifest[key] not in manifest["artifacts"]:
            raise ValueError(f"分类数据包的 {key} 路径或清单被改变。")
    for name, expected_hash in manifest["artifacts"].items():
        file = (root / name).resolve()
        file.relative_to(root)
        if not file.is_file() or digest(file) != expected_hash:
            raise ValueError(f"分类数据包文件缺失或校验失败：{name}")
    if manifest["notebook_definitions_sha256"] != source_snapshot(cfg):
        raise ValueError("Notebook 定义与数据包导出时不同，请使用对应版本或重新导出。")


def require_native_images(cfg: dict) -> None:
    if cfg.get("data_access") == "classification_cache_only":
        raise ValueError("本数据包只有分类缓存，没有原分辨率影像；影像组学和分割需另外准备原生数据。")


In [ ]:
CONFIG_FILE = "config.json"  # 单例检查可改为 config.sample.json
cfg = load_config(NB_ROOT / CONFIG_FILE)
p = paths(cfg)
print("实验：", cfg["run_id"])
print("数据用途：", cfg.get("data_access", "native_images"))
print("结果目录：", p["run"])


## 表格与数组处理

NumPy 用于处理影像数组和预测概率，pandas 用于读取患者清单、合并临床信息及整理结果表。JSON 用于保存参数和指标；正则表达式用于检查编号等文本字段。


In [ ]:
from __future__ import annotations
import json
import re
import numpy as np
import pandas as pd


## 影像读取工具与输入规格

`nibabel` 读取 NIfTI 影像及其空间信息，`zoom` 负责调整切片尺寸，`stats` 用于比较两组患者的基线特征。这里把输入规格设为每位患者 4 层、每层 224 × 224 像素；三个影像通道依次为 T1-FLAIR、T2 和 DWI b1000。


In [ ]:
from __future__ import annotations
import nibabel as nib
from scipy import stats
from scipy.ndimage import zoom
SIZE = 224
SLICES = 4
IMAGES = ("t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz")
CACHE = OUT_DIR = TEMPORAL_OUT = TEMPORAL_SPLIT = None
PREP_BASE = PREP_OUT = ADAPTER_XLSX = None


## 连接输入与输出目录

`configure_baseline` 把配置中确定的文件位置传给后续函数，包括影像缓存、临床表、时间分组表和模型输出目录。后面调用数据处理或模型函数时，就会使用同一组路径。


In [ ]:
from __future__ import annotations

def configure_baseline(cache, full_cv, temporal, adapter, patients_root=None):
    """Configure only input/output locations; do not alter the training recipe."""
    global CACHE, OUT_DIR, TEMPORAL_OUT, TEMPORAL_SPLIT
    global PREP_BASE, PREP_OUT, ADAPTER_XLSX
    CACHE, OUT_DIR, TEMPORAL_OUT = Path(cache), Path(full_cv), Path(temporal)
    TEMPORAL_SPLIT, ADAPTER_XLSX = TEMPORAL_OUT / "temporal_split.csv", Path(adapter)
    PREP_BASE, PREP_OUT = Path(patients_root) if patients_root else None, CACHE


## 四层输入的归一化与缩放方法

下面保留取层解析、影像读取和百分位归一化函数。每位患者从临床表记录的起始层开始，连续取四层；每个序列分别用这四层的第 1 和第 99.5 百分位缩放并截断到 0～1。

影像使用线性插值，标签 1、2、3 拆成三个二值通道后使用最近邻插值，目标尺寸都是 224×224。缓存的 `images` 和 `masks` 形状为 `[患者数, 4, 3, 224, 224]`，结局继续按 MoCA ≤ 22 定义。

这一单元中的 `prepare_cache_main` 保留原批处理过程供对照。下面的实际生成入口读取已经核查的患者清单，并复用相同的归一化函数和插值设置。


In [ ]:
from __future__ import annotations


# 直接复制：code/prep_resnet50_manual.py :: normalize_slices
def normalize_slices(vol: np.ndarray, zs: list[int]) -> np.ndarray:
    vals = vol[:, :, zs]
    lo, hi = np.percentile(vals, [1.0, 99.5])
    if hi <= lo:
        hi = lo + 1e-6
    return np.clip((vals - lo) / (hi - lo), 0.0, 1.0)


# 直接复制：code/prep_resnet50_manual.py :: load_volume
def load_volume(path: Path) -> np.ndarray:
    return np.asarray(nib.load(str(path)).dataobj, dtype=np.float32)


# 直接复制：code/prep_resnet50_manual.py :: parse_max_layer
def parse_max_layer(value) -> int | None:
    text = str(value).strip()
    if text in {"无", "NONE", "NA", ""}:
        return None
    try:
        return int(float(text))
    except ValueError:
        return None


# 直接复制：code/prep_resnet50_manual.py :: main
def prepare_cache_main() -> None:
    df = pd.read_excel(ADAPTER_XLSX, sheet_name="Sheet1", engine="openpyxl")
    df.columns = [str(c).strip() for c in df.columns]
    mr_series = df.iloc[:, 0].astype(str).str.strip().str.upper()
    max_series = df["脑室体素最大层"].map(parse_max_layer)
    moca = pd.to_numeric(df["MOCA"], errors="coerce")

    patient_dirs = {}
    for patient_dir in PREP_BASE.iterdir():
        if not patient_dir.is_dir():
            continue
        match = re.search(r"(MR\d+)$", patient_dir.name.upper())
        if match:
            patient_dirs[match.group(1)] = patient_dir

    images, masks, labels, mrs, max_layers = [], [], [], [], []
    skipped = []

    for idx in range(len(df)):
        if pd.isna(max_series[idx]) or pd.isna(moca[idx]):
            continue
        mr_raw = mr_series[idx]
        digits_match = re.search(r"(\d+)$", mr_raw)
        if not digits_match:
            skipped.append((mr_raw, "no MR number"))
            continue
        mr = f"MR{digits_match.group(1)}"
        patient_dir = patient_dirs.get(mr)
        if patient_dir is None:
            skipped.append((mr, "patient directory not found"))
            continue
        reg = patient_dir / "配准图像"
        mask_path = reg / "Untitled.nii.gz"
        if not mask_path.exists():
            skipped.append((mr, "missing Untitled.nii.gz"))
            continue

        try:
            t1 = load_volume(reg / IMAGES[0])
            t2 = load_volume(reg / IMAGES[1])
            dwi = load_volume(reg / IMAGES[2])
            label_vol = np.asarray(nib.load(str(mask_path)).dataobj, dtype=np.uint8)
        except Exception as exc:
            skipped.append((mr, type(exc).__name__))
            continue

        z0 = int(max_series[idx]) - 1
        zs = [z0, z0 + 1, z0 + 2, z0 + 3]
        if zs[-1] >= t1.shape[2]:
            skipped.append((mr, f"slice overflow: {zs}, shape={t1.shape}"))
            continue

        norm_vols = [
            normalize_slices(vol, zs) for vol in (t1, t2, dwi)
        ]
        h, w = t1.shape[0], t1.shape[1]
        patient_images = np.empty((SLICES, 3, SIZE, SIZE), dtype=np.float16)
        patient_masks = np.zeros((SLICES, 3, SIZE, SIZE), dtype=np.uint8)

        for s, z in enumerate(zs):
            for c, vol in enumerate(norm_vols):
                patient_images[s, c] = zoom(
                    vol[:, :, s],
                    (SIZE / h, SIZE / w),
                    order=1,
                    prefilter=False,
                ).astype(np.float16)

            for label_value in range(1, 4):
                patient_masks[s, label_value - 1] = zoom(
                    (label_vol[:, :, z] == label_value).astype(np.uint8),
                    (SIZE / h, SIZE / w),
                    order=0,
                    prefilter=False,
                )

        images.append(patient_images)
        masks.append(patient_masks)
        labels.append(int(moca[idx] <= 22))
        mrs.append(mr)
        max_layers.append(int(max_series[idx]))

        if len(images) % 40 == 0:
            print(f"processed {len(images)} patients", flush=True)

    images_arr = np.stack(images)
    masks_arr = np.stack(masks)
    labels_arr = np.asarray(labels, dtype=np.int64)
    max_layers_arr = np.asarray(max_layers, dtype=np.int16)

    np.savez_compressed(
        PREP_OUT,
        images=images_arr,
        masks=masks_arr,
        labels=labels_arr,
        mrs=np.asarray(mrs),
        max_layers=max_layers_arr,
    )

    print("\npatients", len(images))
    print("images", images_arr.shape, images_arr.dtype)
    print("masks", masks_arr.shape, masks_arr.dtype)
    print("labels", int(labels_arr.sum()), "positive /", int((1 - labels_arr).sum()), "negative")
    print("skipped", skipped)
    print("wrote", PREP_OUT)


## 按检查时间分组，并比较基线特征

`prepare_temporal_main` 将缓存中的患者与临床表匹配，按检查时间、MR 编号排序，以第 144 位患者的检查日期为分界。同一天的患者全部进入开发队列，日期更晚的患者进入验证队列。

分组后，代码比较两组的年龄、教育年限、NIHSS、MoCA、Fazekas、性别及病灶像素数等特征。连续变量以中位数和四分位数表示，并进行 Mann–Whitney U 检验；二分类变量比较比例，同时计算标准化差异（SMD），帮助判断两组人群是否存在明显差别。

这里的病灶计数来自缩放后的四层掩模，单位是缓存中的像素数，不能直接解释为完整病灶的物理体积。


In [ ]:
from __future__ import annotations


# 直接复制：code/prepare_temporal_validation.py :: parse_time
def parse_time(value) -> pd.Timestamp | pd.NaT:
    return pd.to_datetime(str(value).strip(), format="%Y.%m.%d", errors="coerce")


# 直接复制：code/prepare_temporal_validation.py :: format_median_iqr
def format_median_iqr(series: pd.Series) -> str:
    return f"{series.median():.2f} ({series.quantile(0.25):.2f}-{series.quantile(0.75):.2f})"


# 直接复制：code/prepare_temporal_validation.py :: smd_continuous
def smd_continuous(a: pd.Series, b: pd.Series) -> float:
    pooled = np.sqrt((a.std(ddof=1) ** 2 + b.std(ddof=1) ** 2) / 2.0)
    if pooled == 0:
        return 0.0
    return float((a.mean() - b.mean()) / pooled)


# 直接复制：code/prepare_temporal_validation.py :: smd_binary
def smd_binary(a: pd.Series, b: pd.Series) -> float:
    p1 = float(a.mean())
    p2 = float(b.mean())
    denom = np.sqrt((p1 * (1 - p1) + p2 * (1 - p2)) / 2.0)
    if denom == 0:
        return 0.0
    return float((p1 - p2) / denom)


# 直接复制：code/prepare_temporal_validation.py :: main
def prepare_temporal_main() -> None:
    TEMPORAL_OUT.mkdir(parents=True, exist_ok=True)
    excel = pd.read_excel(ADAPTER_XLSX, sheet_name="Sheet1", engine="openpyxl")
    excel.columns = [str(c).strip() for c in excel.columns]
    excel["mr"] = excel.iloc[:, 1].astype(str).str.strip().str.upper()
    excel["check_time"] = excel["检查时间"].map(parse_time)

    cache = np.load(CACHE, allow_pickle=True)
    cache_mrs = np.asarray([str(x).upper() for x in cache["mrs"]])
    cohort = excel[excel["mr"].isin(set(cache_mrs))].copy()
    assert len(cohort) == 206 and cohort["mr"].is_unique
    cohort = cohort.sort_values(["check_time", "mr"], kind="mergesort").reset_index(drop=True)
    assert cohort["check_time"].notna().all()

    # All patients on the 70% boundary date stay in development.
    boundary_date = cohort.loc[143, "check_time"]
    cohort["split"] = np.where(cohort["check_time"] <= boundary_date, "development", "validation")
    split_df = cohort[["mr", "check_time", "split"]].rename(columns={"check_time": "check_time"})
    split_df.to_csv(TEMPORAL_OUT / "temporal_split.csv", index=False)

    dev = cohort[cohort["split"] == "development"].copy()
    val = cohort[cohort["split"] == "validation"].copy()
    print("split", len(dev), len(val), "boundary", boundary_date.date())
    print("outcome", dev["MOCA"].le(22).sum(), val["MOCA"].le(22).sum())

    dev_idx = {mr: i for i, mr in enumerate(cache_mrs)}
    lesion = np.asarray(cache["masks"], dtype=np.int32)
    for df in (dev, val):
        idx = np.asarray([dev_idx[mr] for mr in df["mr"]], dtype=int)
        for label in (1, 2, 3):
            df[f"lesion_{label}_vox"] = lesion[idx, :, label - 1, :, :].sum(axis=(1, 2, 3))

    dev["male"] = (dev["性别"] == "男").astype(int)
    val["male"] = (val["性别"] == "男").astype(int)
    dev["impaired"] = (dev["MOCA"] <= 22).astype(int)
    val["impaired"] = (val["MOCA"] <= 22).astype(int)

    rows = []
    for name in ["age", "fazekas", "education", "nihss", "moca"]:
        col_map = {
            "age": "年龄",
            "fazekas": "Fazekas",
            "education": "受教育年限",
            "nihss": "NIHSS",
            "moca": "MOCA",
        }
        a = pd.to_numeric(dev[col_map[name]], errors="coerce")
        b = pd.to_numeric(val[col_map[name]], errors="coerce")
        a_test = a.dropna()
        b_test = b.dropna()
        rows.append(
            {
                "variable": name,
                "development": format_median_iqr(a),
                "validation": format_median_iqr(b),
                "p": stats.mannwhitneyu(a_test, b_test, alternative="two-sided").pvalue,
                "smd": smd_continuous(a_test, b_test),
            }
        )

    for name, a, b in [
        ("male", dev["male"], val["male"]),
        ("impaired", dev["impaired"], val["impaired"]),
    ]:
        table = pd.crosstab(
            pd.Series(np.r_[np.zeros(len(a)), np.ones(len(b))], dtype=int),
            pd.Series(np.r_[a.values, b.values], dtype=int),
        )
        if table.shape == (2, 2):
            p = stats.chi2_contingency(table, correction=False).pvalue
        else:
            p = np.nan
        rows.append(
            {
                "variable": name,
                "development": f"{int(a.sum())}/{len(a)}",
                "validation": f"{int(b.sum())}/{len(b)}",
                "p": p,
                "smd": smd_binary(a, b),
            }
        )

    for label in (1, 2, 3):
        a = dev[f"lesion_{label}_vox"]
        b = val[f"lesion_{label}_vox"]
        rows.append(
            {
                "variable": f"lesion_{label}_vox",
                "development": format_median_iqr(a),
                "validation": format_median_iqr(b),
                "p": stats.mannwhitneyu(a, b, alternative="two-sided").pvalue,
                "smd": smd_continuous(a, b),
            }
        )

    baseline = pd.DataFrame(rows)
    baseline["p"] = baseline["p"].map(lambda v: f"{v:.4f}")
    baseline["smd"] = baseline["smd"].map(lambda v: f"{v:.3f}")
    baseline.to_csv(TEMPORAL_OUT / "baseline_comparison.csv", index=False)
    print(baseline.to_string(index=False))

    with open(TEMPORAL_OUT / "temporal_split_summary.md", "w", encoding="utf-8") as f:
        f.write("# 时间分组与基线比较\n\n")
        f.write(f"- 开发队列：{len(dev)} 例\n")
        f.write(f"- 独立验证队列：{len(val)} 例\n")
        f.write(f"- 时间边界：{boundary_date.date()}\n\n")
        f.write(baseline.to_string(index=False))

    print("wrote", TEMPORAL_OUT / "temporal_split.csv", TEMPORAL_OUT / "baseline_comparison.csv")


## 临床表整理与取层记录

这一组函数读取真实表头，将“影像号”统一成 MR 编号，并整理 MoCA、检查时间和五个临床变量。编号重复、日期错误会写入问题表；缺失或无效 MoCA 会单独记录。取层必须是从 1 开始的整数，避免把 11.5 这样的异常记录截成第 11 层。

临床工作表保留 Excel 原行号，方便回查。`preparation_paths` 将样例检查结果放入独立的 `data/sample/`，与全量缓存分开。


In [ ]:
from __future__ import annotations
import json
import re
from pathlib import Path
import numpy as np
import pandas as pd


def preparation_paths(cfg):
    mode = cfg.get("preprocessing", {}).get("mode", "full")
    if mode not in {"full", "sample"}:
        raise ValueError("preprocessing.mode 只能是 full 或 sample。")
    p = paths(cfg)
    root = p["data"] if mode == "full" else p["data"] / "sample"
    return {"root": root, "mode": mode, "matching": root / "patient_matching.csv",
            "audit": root / "volume_audit.csv", "audit_details": root / "volume_audit_details.json",
            "cache": p["cache"] if mode == "full" else root / "resnet50_manual_cache_sample.npz",
            "cache_manifest": root / "cache_manifest.json"}


def preparation_settings(cfg):
    """Only settings which determine patient selection or audit interpretation."""
    settings = cfg.get("preprocessing", {})
    return {"mode": preparation_paths(cfg)["mode"], "sample_mrs": settings.get("sample_mrs", []),
            "expected_orientation": settings.get("expected_orientation", "RAS"),
            "audit_dicom_geometry": settings.get("audit_dicom_geometry", True),
            "annotation_scope": cfg.get("annotation_scope", "unconfirmed"),
            "annotation_scope_by_mr": settings.get("annotation_scope_by_mr", {}),
            "expected_patients": int(cfg["expected_patients"])}


def excel_headers(cfg):
    return pd.read_excel(require_input(cfg, "excel_path"), sheet_name=cfg["sheet_name"], nrows=0).columns.tolist()


def normalized_mr(value) -> str:
    if pd.isna(value) or isinstance(value, (bool, np.bool_)):
        raise ValueError("缺少有效影像号。")
    if isinstance(value, (int, float, np.integer, np.floating)):
        if not np.isfinite(value) or value < 0 or not float(value).is_integer():
            raise ValueError(f"影像号应为非负整数或 MR数字：{value!r}")
        value = str(int(value))
    match = re.fullmatch(r"(?:MR\s*)?(\d+)", str(value).strip().upper())
    if not match:
        raise ValueError(f"影像号格式无法确定：{value!r}；请使用 MR数字 或纯数字。")
    return "MR" + match.group(1)


def four_slice_indices(value, depth=None):
    try:
        number = float(value)
    except (ValueError, TypeError):
        raise ValueError(f"取层记录不是数值：{value!r}") from None
    if not np.isfinite(number) or not number.is_integer() or number < 1:
        raise ValueError(f"起始层必须是从1开始的整数：{value!r}")
    start = int(number)
    if start > np.iinfo(np.int16).max:
        raise ValueError("起始层超出缓存支持范围。")
    zs = list(range(start - 1, start + 3))
    if depth is not None and zs[-1] >= depth:
        raise ValueError(f"所选第{start}～{start+3}层超出总层数{depth}。")
    return zs


def prepare_adapter(cfg):
    if cfg.get("data_access") == "classification_cache_only":
        raise ValueError("当前是服务器分类数据包，请直接使用已有缓存和清单。")
    p = paths(cfg)
    fresh(p["adapter"], p["data"] / "clinical_adapter_meta.json")
    source_path = require_input(cfg, "excel_path")
    source_hash = digest(source_path)
    source = pd.read_excel(source_path, sheet_name=cfg["sheet_name"])
    source.columns = source.columns.map(lambda x: str(x).strip())
    if not source.columns.is_unique:
        raise ValueError("Excel 列名去空格后重复。")
    columns = cfg["columns"]
    missing = [k for k, name in columns.items() if name is None or name not in source.columns]
    if missing:
        raise ValueError(f"列映射与真实表头不符：{missing}；现有列：{source.columns.tolist()}")
    p["data"].mkdir(parents=True, exist_ok=True)
    issues, mrs = [], []
    for i, value in enumerate(source[columns["mr"]]):
        try:
            mrs.append(normalized_mr(value))
        except ValueError as exc:
            mrs.append(None)
            issues.append({"excel_row": i + 2, "field": "mr", "reason": str(exc)})
    duplicate = pd.Series(mrs).notna() & pd.Series(mrs).duplicated(keep=False)
    for i in np.flatnonzero(duplicate):
        issues.append({"excel_row": int(i) + 2, "field": "mr", "reason": f"重复患者编号 {mrs[i]}"})
    moca = pd.to_numeric(source[columns["moca"]], errors="coerce")
    valid_outcome = moca.notna() & np.isfinite(moca) & moca.between(0, 30)
    excluded = pd.DataFrame({"excel_row": np.arange(len(source)) + 2, "mr": mrs,
                             "reason": "missing_or_invalid_moca"}).loc[~valid_outcome]
    excluded.to_csv(p["data"] / "excluded_missing_outcome.csv", index=False)
    date_values = source[columns["check_time"]]
    dates = pd.to_datetime(date_values.astype(str).str.strip().str.replace(".", "-", regex=False),
                           format="mixed", errors="coerce")
    for i in np.flatnonzero(valid_outcome & dates.isna()):
        issues.append({"excel_row": int(i) + 2, "field": "check_time", "reason": "无法解析检查日期"})
    pd.DataFrame(issues, columns=["excel_row", "field", "reason"]).to_csv(p["data"] / "clinical_issues.csv", index=False)
    if issues:
        raise ValueError("临床表存在编号或日期问题，请查看 data/clinical_issues.csv。")
    if not valid_outcome.any():
        raise ValueError("没有可用的 MoCA 结局。")
    names = [f"_unused_{i}" for i in range(33)]
    spec = {0: ("MR", "mr"), 1: ("MR_join", "mr"), 2: ("检查时间", "check_time"),
            3: ("性别", "sex"), 4: ("年龄", "age"), 5: ("Fazekas", "fazekas"),
            6: ("脑室体素最大层", "max_layer"), 13: ("受教育年限", "education"),
            23: ("NIHSS", "nihss"), 24: ("MOCA", "moca"), 25: ("同型半胱氨酸", "hcy"), 32: ("尿酸", "ua")}
    for i, (name, _) in spec.items():
        names[i] = name
    adapter = pd.DataFrame(index=source.index, columns=names)
    numeric = {"age", "fazekas", "education", "nihss", "hcy", "ua", "moca"}
    for _, (name, key) in spec.items():
        value = pd.Series(mrs, index=source.index) if key == "mr" else source[columns[key]]
        adapter[name] = pd.to_numeric(value, errors="coerce").replace([np.inf, -np.inf], np.nan) if key in numeric else value
    adapter["检查时间"] = dates.dt.strftime("%Y.%m.%d")
    adapter["_excel_row"] = np.arange(len(source)) + 2
    adapter = adapter.loc[valid_outcome].reset_index(drop=True)
    if digest(source_path) != source_hash:
        raise ValueError("读取期间源 Excel 发生变化，请重新准备临床工作表。")
    adapter.to_excel(p["adapter"], sheet_name="Sheet1", index=False)
    dump_json(p["data"] / "clinical_adapter_meta.json", {"source_excel_sha256": source_hash,
        "sheet_name": cfg["sheet_name"], "columns": columns, "adapter_sha256": digest(p["adapter"]),
        "source_rows": len(source), "included_rows": len(adapter), "excluded_outcome_rows": len(excluded)})
    record(cfg, "adapter", {"rows": len(adapter), "missing_outcome_exclusions": len(excluded)})
    return adapter


def read_adapter(cfg):
    p = paths(cfg)
    meta = json.loads((p["data"] / "clinical_adapter_meta.json").read_text(encoding="utf-8"))
    if (digest(p["adapter"]) != meta["adapter_sha256"] or cfg["columns"] != meta["columns"]
            or cfg["sheet_name"] != meta["sheet_name"]):
        raise ValueError("临床工作表或列映射在准备后发生变化，请核对实验目录。")
    if digest(require_input(cfg, "excel_path")) != meta["source_excel_sha256"]:
        raise ValueError("源 Excel 已更新，请使用新 run_id 重新准备数据。")
    adapter = pd.read_excel(p["adapter"], dtype={"MR": str, "MR_join": str})
    if not adapter.MR.is_unique:
        raise ValueError("临床工作表患者编号重复。")
    return adapter



## 按影像号匹配患者目录

`patient_directories` 在患者根目录下寻找以 MR 编号结尾的文件夹。`match_patients` 将这些目录与临床表逐例对应，检查三序列影像和 `Untitled.nii.gz` 是否齐全。

结果会列出临床表找不到的影像、影像目录找不到的临床记录，以及重复目录。全量处理按临床表顺序保存患者，后面的缓存和分组都通过 MR 编号对应。


In [ ]:
from __future__ import annotations

def patient_directories(cfg):
    result, duplicates = {}, []
    for directory in sorted(require_input(cfg, "patients_root").iterdir()):
        match = re.search(r"MR\s*(\d+)$", directory.name.strip().upper())
        if directory.is_dir() and match:
            mr = normalized_mr("MR" + match.group(1))
            if mr in result:
                duplicates.append({"mr": mr, "first": str(result[mr].parent), "second": str(directory)})
            else:
                result[mr] = directory.resolve() / "配准图像"
    prep = preparation_paths(cfg)
    prep["root"].mkdir(parents=True, exist_ok=True)
    pd.DataFrame(duplicates, columns=["mr", "first", "second"]).to_csv(prep["root"] / "duplicate_directories.csv", index=False)
    if duplicates:
        raise ValueError("发现重复 MR 文件夹，请查看 duplicate_directories.csv。")
    return result


def match_patients(cfg):
    adapter, directories = read_adapter(cfg), patient_directories(cfg)
    prep = preparation_paths(cfg)
    selected_ids = set(adapter.MR)
    if prep["mode"] == "sample":
        requested = cfg.get("preprocessing", {}).get("sample_mrs", [])
        if not requested:
            raise ValueError("单例/小样本检查请填写 preprocessing.sample_mrs。")
        selected_ids = {normalized_mr(mr) for mr in requested}
        if not selected_ids <= set(adapter.MR):
            raise ValueError(f"样例编号在临床表中不存在：{sorted(selected_ids-set(adapter.MR))}")
    rows = []
    for _, row in adapter.iterrows():
        mr, directory = row.MR, directories.get(row.MR)
        missing = [name for name in IMAGE_FILES + ["Untitled.nii.gz"] if directory is None or not (directory / name).is_file()]
        reason = "missing_patient_directory" if directory is None else ("missing_files: " + ", ".join(missing) if missing else "")
        rows.append({"mr": mr, "excel_row": int(row["_excel_row"]), "selected": mr in selected_ids,
                     "matched": not missing, "reg_dir": str(directory) if directory else "", "reason": reason})
    table = pd.DataFrame(rows)
    table.to_csv(prep["matching"], index=False)
    pd.DataFrame([{"mr": mr, "reg_dir": str(directory)} for mr, directory in directories.items() if mr not in set(adapter.MR)],
                 columns=["mr", "reg_dir"]).to_csv(prep["root"] / "directories_without_clinical.csv", index=False)
    dump_json(prep["root"] / "matching_meta.json", {"adapter_sha256": digest(paths(cfg)["adapter"]),
        "matching_sha256": digest(prep["matching"]), "settings": preparation_settings(cfg),
        "patients_root": str(require_input(cfg, "patients_root").resolve())})
    return table


def read_matching(cfg):
    prep = preparation_paths(cfg)
    meta = json.loads((prep["root"] / "matching_meta.json").read_text(encoding="utf-8"))
    read_adapter(cfg)
    if (meta["settings"] != preparation_settings(cfg) or meta["adapter_sha256"] != digest(paths(cfg)["adapter"])
            or meta["matching_sha256"] != digest(prep["matching"])
            or meta["patients_root"] != str(require_input(cfg, "patients_root").resolve())):
        raise ValueError("患者匹配依据发生变化，请重新执行患者匹配和影像检查。")
    table = pd.read_csv(prep["matching"], dtype={"mr": str}, keep_default_na=False)
    return table.loc[table.selected.astype(str).str.lower().eq("true")].reset_index(drop=True)



## 检查切片、标签和空间信息

`inspect_patient` 核对三序列与人工掩模的尺寸、空间变换、方向、有效数值和标签集合，并检查四层是否越界。它还记录每类标签在选定四层内的体素数、四层外是否有标签，以及归一化使用的百分位数。

`dicom_geometry` 只读取 DICOM 头信息，通过相邻切片的位置计算层中心间距，和 NIfTI 的间距并列检查。单位未声明、层间距不同或缺少 DICOM 时，会记录待核对项，保留现有影像。

样例 MR251203027 已确认只标注四层；其他患者的标注范围由配置中的 `annotation_scope_by_mr` 单独记录，未知的保持 `unconfirmed`。空白层不能据此认定没有病灶。


In [ ]:
from __future__ import annotations

def dicom_geometry(patient_dir):
    result = {}
    folders = {"t1": "T1_FLAIR_AX", "t2": "T2_PROPELLER", "dwi": "DWI_b1000"}
    try:
        import pydicom
    except ImportError:
        return {"status": "not_checked", "reason": "缺少 pydicom，未核对原始 DICOM。", "series": result}
    tags = ["Rows", "Columns", "PixelSpacing", "SliceThickness", "SpacingBetweenSlices",
            "ImagePositionPatient", "ImageOrientationPatient", "NumberOfFrames"]
    for sequence, name in folders.items():
        files = sorted((Path(patient_dir) / name).glob("*.[dD][cC][mM]"))
        if not files:
            result[sequence] = {"status": "not_available"}
            continue
        try:
            datasets = [pydicom.dcmread(str(f), stop_before_pixels=True, specific_tags=tags) for f in files]
            if any(int(getattr(d, "NumberOfFrames", 1)) != 1 for d in datasets):
                raise ValueError("多帧 DICOM 需要单独解析。")
            orientations = np.asarray([d.ImageOrientationPatient for d in datasets], dtype=float)
            positions = np.asarray([d.ImagePositionPatient for d in datasets], dtype=float)
            if not np.isfinite(orientations).all() or not np.isfinite(positions).all():
                raise ValueError("DICOM 方向或位置包含无效数值。")
            if not np.allclose(orientations, orientations[0], atol=1e-4, rtol=0):
                raise ValueError("同一序列内切片方向不一致。")
            normal = np.cross(orientations[0, :3], orientations[0, 3:])
            if np.linalg.norm(normal) < 1e-8:
                raise ValueError("DICOM 切片方向无效。")
            positions_along_normal = np.sort(positions @ (normal / np.linalg.norm(normal)))
            distance = np.diff(positions_along_normal)
            if not len(distance) or np.any(distance < 1e-3):
                raise ValueError("切片位置重复或不足两层。")
            pixel_spacing = np.asarray(datasets[0].PixelSpacing, dtype=float)
            thickness = float(datasets[0].SliceThickness) if hasattr(datasets[0], "SliceThickness") else None
            declared_spacing = float(datasets[0].SpacingBetweenSlices) if hasattr(datasets[0], "SpacingBetweenSlices") else None
            if pixel_spacing.shape != (2,) or not np.isfinite(pixel_spacing).all() or np.any(pixel_spacing <= 0):
                raise ValueError("DICOM 像素间距无效。")
            if any(value is not None and not np.isfinite(value) for value in (thickness, declared_spacing)):
                raise ValueError("DICOM 层厚或间距包含无效数值。")
            result[sequence] = {"status": "checked", "files": len(files),
                "shape_2d": [int(datasets[0].Rows), int(datasets[0].Columns)],
                "pixel_spacing_mm": pixel_spacing.tolist(),
                "slice_thickness_mm": thickness,
                "declared_slice_spacing_mm": declared_spacing,
                "measured_slice_spacing_mm": float(np.median(distance)),
                "spacing_range_mm": [float(distance.min()), float(distance.max())]}
        except (OSError, EOFError, ValueError, TypeError, AttributeError, KeyError, pydicom.errors.InvalidDicomError) as exc:
            result[sequence] = {"status": "not_checked", "reason": str(exc)}
    return {"status": "complete" if all(v["status"] == "checked" for v in result.values()) else "incomplete", "series": result}


def inspect_patient(cfg, patient, directory):
    import nibabel as nib
    directory = Path(directory)
    names = IMAGE_FILES + ["Untitled.nii.gz"]
    hashes_before = {name: digest(directory / name) for name in names}
    volumes = [nib.load(str(directory / name)) for name in names]
    ref = volumes[0]
    if len(ref.shape) != 3 or min(ref.shape) < 1 or min(ref.shape[:2]) < 2:
        raise ValueError(f"需要三维影像，实际形状为 {ref.shape}。")
    if not np.isfinite(ref.affine).all() or abs(np.linalg.det(ref.affine[:3, :3])) < 1e-12:
        raise ValueError("参考影像的空间变换无效。")
    if any(v.shape != ref.shape or not np.allclose(v.affine, ref.affine, atol=1e-4, rtol=0) for v in volumes[1:]):
        raise ValueError("三序列影像与人工标注的 shape/affine 不一致。")
    orientation = "".join(nib.aff2axcodes(ref.affine))
    expected_orientation = cfg.get("preprocessing", {}).get("expected_orientation", "RAS")
    if expected_orientation and orientation != expected_orientation:
        raise ValueError(f"影像方向为 {orientation}，与配置的 {expected_orientation} 不同；需人工核对，不自动翻转。")
    zs = four_slice_indices(patient["脑室体素最大层"], ref.shape[2])
    arrays = [np.asarray(v.dataobj) for v in volumes]
    if any(not np.isfinite(a).all() for a in arrays):
        raise ValueError("影像或标签包含 NaN/Inf。")
    label_values, counts = np.unique(arrays[-1], return_counts=True)
    if not set(label_values) <= {0, 1, 2, 3}:
        raise ValueError(f"人工标签需要0/1/2/3，实际为 {label_values.tolist()}。")
    geometry_warnings, ranges, files = [], {}, {}
    for name, volume, array in zip(names, volumes, arrays):
        units = volume.header.get_xyzt_units()[0]
        spacing = np.asarray(volume.header.get_zooms()[:3], dtype=float)
        if not np.isfinite(spacing).all() or np.any(spacing <= 0):
            raise ValueError(f"{name} 的体素间距无效。")
        if units != "mm":
            geometry_warnings.append(f"{name}:空间单位为 {units}，物理尺度待核对")
        qform, qcode = volume.get_qform(coded=True)
        sform, scode = volume.get_sform(coded=True)
        if qcode and scode and not np.allclose(qform, sform, atol=1e-3, rtol=0):
            geometry_warnings.append(f"{name}:qform 与 sform 不一致")
        affine_spacing = np.linalg.norm(volume.affine[:3, :3], axis=0)
        if not np.allclose(spacing, affine_spacing, atol=1e-4, rtol=1e-4):
            geometry_warnings.append(f"{name}:头文件体素间距与空间变换不一致")
        files[name] = {"sha256": hashes_before[name], "bytes": (directory / name).stat().st_size,
                       "shape": list(volume.shape), "spacing": spacing.tolist(), "spatial_unit": units,
                       "affine": volume.affine.tolist(), "orientation": "".join(nib.aff2axcodes(volume.affine)),
                       "qform_code": int(qcode), "sform_code": int(scode)}
        if name in IMAGE_FILES:
            with np.errstate(over="ignore", invalid="ignore"):
                image = array.astype(np.float32)
            if not np.isfinite(image).all():
                raise ValueError(f"{name} 转为 float32 后出现无效值。")
            lo, hi = np.percentile(image[:, :, zs], [1, 99.5])
            ranges[name] = {"percentile_1": float(lo), "percentile_99_5": float(hi)}
    warnings = [f"{name}:选定四层的强度范围退化，归一化将沿用原代码的微小分母处理"
                for name, limits in ranges.items() if limits["percentile_99_5"] <= limits["percentile_1"]]
    labels = arrays[-1]
    selected_mask = labels[:, :, zs]
    scope = cfg.get("preprocessing", {}).get("annotation_scope_by_mr", {}).get(str(patient.MR), cfg.get("annotation_scope", "unconfirmed"))
    if scope not in {"selected_slices", "full_volume", "unconfirmed"}:
        raise ValueError(f"{patient.MR} 的标注范围配置无效：{scope}")
    outside = int(np.count_nonzero(labels) - np.count_nonzero(selected_mask))
    if scope == "selected_slices" and outside:
        warnings.append("选定四层外仍有标签，请核对标注范围记录")
    if not np.count_nonzero(selected_mask):
        warnings.append("选定四层的人工标签全空，请核对；不据此自动排除患者")
    dicom = dicom_geometry(directory.parent) if cfg.get("preprocessing", {}).get("audit_dicom_geometry", True) else {"status": "not_requested", "series": {}}
    if dicom["status"] != "complete":
        geometry_warnings.append("原始 DICOM 空间信息未完成全部核对")
    for sequence, name in zip(("t1", "t2", "dwi"), IMAGE_FILES):
        geometry = dicom["series"].get(sequence, {})
        if geometry.get("status") == "checked":
            if not np.isclose(geometry["measured_slice_spacing_mm"], files[name]["spacing"][2], atol=0.05, rtol=0):
                geometry_warnings.append(f"{name}:DICOM层中心间距{geometry['measured_slice_spacing_mm']:.4f}，NIfTI第三轴间距{files[name]['spacing'][2]:.4f}，需核查转换/重采样记录")
            if geometry["spacing_range_mm"][1] - geometry["spacing_range_mm"][0] > 0.05:
                geometry_warnings.append(f"{name}:DICOM 各层中心间距不均匀")
    if hashes_before != {name: digest(directory / name) for name in names}:
        raise ValueError("影像检查期间源文件发生变化，请重新检查。")
    return {"mr": str(patient.MR), "files": files, "selected_slices_1based": [z + 1 for z in zs],
            "annotation_scope": scope, "nonzero_label_slices_1based": (np.flatnonzero((labels != 0).any(axis=(0, 1))) + 1).tolist(),
            "label_counts_native": {str(int(v)): int(c) for v, c in zip(label_values, counts)},
            "label_counts_selected": {str(k): int(np.count_nonzero(selected_mask == k)) for k in (1, 2, 3)},
            "foreground_outside_selected": outside, "normalization": ranges,
            "dicom_geometry": dicom, "warnings": geometry_warnings + warnings,
            "physical_geometry_needs_review": bool(geometry_warnings)}



## 汇总检查结果并核对文件版本

`audit_volumes` 为每位选中患者生成一行结果。`eligible` 表示是否满足当前四层分类输入要求；`reason` 是阻止生成缓存的问题；`warnings` 和 `physical_geometry_needs_review` 记录仍需检查的空间信息。

`checked_inputs` 在生成缓存前核对临床表、匹配表、核查记录和影像文件的校验值。文件修改后要重新检查，避免使用已经过期的核查结果。


In [ ]:
from __future__ import annotations

def audit_volumes(cfg):
    import nibabel as nib
    prep, matching = preparation_paths(cfg), read_matching(cfg)
    adapter = read_adapter(cfg).set_index("MR", drop=False)
    rows, details = [], {}
    for _, match in matching.iterrows():
        row = {"mr": match.mr, "eligible": False, "reason": "", "warnings": "",
               "physical_geometry_needs_review": True, "annotation_scope": "unconfirmed"}
        try:
            if str(match.matched).lower() != "true":
                raise ValueError(match.reason)
            info = inspect_patient(cfg, adapter.loc[match.mr], match.reg_dir)
            details[match.mr] = info
            reference = info["files"][IMAGE_FILES[0]]
            row.update(eligible=True, warnings="；".join(info["warnings"]),
                physical_geometry_needs_review=info["physical_geometry_needs_review"], annotation_scope=info["annotation_scope"],
                shape=json.dumps(reference["shape"]), spacing=json.dumps(reference["spacing"]), orientation=reference["orientation"],
                selected_slices_1based=json.dumps(info["selected_slices_1based"]),
                label_1_voxels=info["label_counts_selected"]["1"], label_2_voxels=info["label_counts_selected"]["2"],
                label_3_voxels=info["label_counts_selected"]["3"], foreground_outside_selected=info["foreground_outside_selected"])
        except (OSError, EOFError, ValueError, TypeError, KeyError, nib.filebasedimages.ImageFileError) as exc:
            row["reason"] = str(exc)
        rows.append(row)
    audit = pd.DataFrame(rows)
    audit.to_csv(prep["audit"], index=False)
    dump_json(prep["audit_details"], {"adapter_sha256": digest(paths(cfg)["adapter"]),
        "matching_sha256": digest(prep["matching"]), "audit_sha256": digest(prep["audit"]),
        "settings": preparation_settings(cfg), "notebook_definitions_sha256": source_snapshot(cfg), "patients": details})
    return audit


def checked_inputs(cfg):
    prep, matching = preparation_paths(cfg), read_matching(cfg)
    audit = pd.read_csv(prep["audit"], dtype={"mr": str}, keep_default_na=False)
    details = json.loads(prep["audit_details"].read_text(encoding="utf-8"))
    if (details["adapter_sha256"] != digest(paths(cfg)["adapter"]) or details["matching_sha256"] != digest(prep["matching"])
            or details["audit_sha256"] != digest(prep["audit"]) or details["settings"] != preparation_settings(cfg)
            or details["notebook_definitions_sha256"] != source_snapshot(cfg)):
        raise ValueError("核查记录的输入、设置或代码定义已变化，请重新检查影像。")
    if not audit.mr.is_unique or audit.mr.tolist() != matching.mr.tolist():
        raise ValueError("核查表与本次选定患者不一致。")
    if not audit.eligible.astype(str).str.lower().eq("true").all():
        raise ValueError("存在未通过检查的患者，请先处理 volume_audit.csv 的 reason；不静默跳过。")
    if prep["mode"] == "full" and len(matching) != int(cfg["expected_patients"]):
        raise ValueError(f"全量模式需要{cfg['expected_patients']}例，当前{len(matching)}例；样例检查请使用 sample 模式。")
    if not len(matching) or set(details["patients"]) != set(matching.mr):
        raise ValueError("核查明细缺少患者。")
    for _, row in matching.iterrows():
        for filename, info in details["patients"][row.mr]["files"].items():
            if digest(Path(row.reg_dir) / filename) != info["sha256"]:
                raise ValueError(f"{row.mr}/{filename} 在核查后发生变化，请重新检查。")
    return matching, read_adapter(cfg).set_index("MR", drop=False), details



## 生成缓存或核对已有缓存

`cache_patient` 对每位患者取连续四层，沿用 1%～99.5% 归一化、影像线性插值和掩模最近邻插值，输出 224×224 的输入。影像保存为 float16，掩模为 uint8。

`run_original_preprocessing` 只处理已通过检查的患者，不再静默跳过异常病例。保存后会检查数组形状、患者顺序、结局和取层记录，再写入 `cache_manifest.json`。中断产生的 `.partial.npz` 会保留，完整文件只在检查成功后发布。

如果已有缓存，`verify_existing_cache` 会逐患者重新计算输入并与缓存比较。这一步需要原生影像，适合在数据所在机器上完成一次。


In [ ]:
from __future__ import annotations

def cache_patient(directory, max_layer):
    """Same percentile normalization and scipy zoom calls as the original code."""
    import nibabel as nib
    from scipy.ndimage import zoom
    directory = Path(directory)
    volumes = [load_volume(directory / name) for name in IMAGE_FILES]
    label_volume = np.asarray(nib.load(str(directory / "Untitled.nii.gz")).dataobj)
    if any(v.shape != volumes[0].shape for v in volumes) or label_volume.shape != volumes[0].shape:
        raise ValueError("缓存读取时影像形状发生变化。")
    if any(not np.isfinite(v).all() for v in volumes) or not np.isfinite(label_volume).all():
        raise ValueError("缓存读取时发现非有限值。")
    if not set(np.unique(label_volume)) <= {0, 1, 2, 3}:
        raise ValueError("人工标签值不合法。")
    label_volume = label_volume.astype(np.uint8)
    zs = four_slice_indices(max_layer, volumes[0].shape[2])
    norm_volumes = [normalize_slices(volume, zs) for volume in volumes]
    h, w = volumes[0].shape[:2]
    images = np.empty((4, 3, 224, 224), dtype=np.float16)
    masks = np.zeros((4, 3, 224, 224), dtype=np.uint8)
    for s, z in enumerate(zs):
        for c, volume in enumerate(norm_volumes):
            images[s, c] = zoom(volume[:, :, s], (224 / h, 224 / w), order=1, prefilter=False).astype(np.float16)
        for label in range(1, 4):
            masks[s, label - 1] = zoom((label_volume[:, :, z] == label).astype(np.uint8),
                                     (224 / h, 224 / w), order=0, prefilter=False)
    return images, masks


def validate_cache_file(cache_path, expected_mrs=None, expected_labels=None, expected_layers=None):
    with np.load(cache_path, allow_pickle=False) as cache:
        required = {"images", "masks", "labels", "mrs", "max_layers"}
        if not required <= set(cache.files):
            raise ValueError(f"缓存缺字段：{sorted(required-set(cache.files))}")
        images, masks = cache["images"], cache["masks"]
        mrs, labels, layers = cache["mrs"].astype(str), cache["labels"], cache["max_layers"]
        if mrs.ndim != 1 or len(set(mrs)) != len(mrs) or any(normalized_mr(x) != x for x in mrs):
            raise ValueError("缓存编号重复、维度错误或格式不统一。")
        shape = (len(mrs), 4, 3, 224, 224)
        if images.shape != shape or masks.shape != shape or labels.shape != (len(mrs),) or layers.shape != (len(mrs),):
            raise ValueError("缓存形状不符合四层三通道格式。")
        if images.dtype != np.float16 or masks.dtype != np.uint8 or labels.dtype != np.int64 or layers.dtype != np.int16:
            raise ValueError("缓存数据类型与原格式不一致。")
        if not np.isfinite(images).all() or np.any((images < 0) | (images > 1)) or not set(np.unique(masks)) <= {0, 1}:
            raise ValueError("缓存影像需在[0,1]，掩模需为二值。")
        if not set(labels) <= {0, 1} or np.any(layers < 1):
            raise ValueError("缓存结局或取层记录无效。")
        for actual, expected, name in ((mrs, expected_mrs, "患者顺序"), (labels, expected_labels, "结局"), (layers, expected_layers, "取层")):
            if expected is not None and not np.array_equal(actual, np.asarray(expected)):
                raise ValueError(f"缓存{name}与当前临床和核查记录不一致。")
        return {"patients": len(mrs), "shape": list(shape), "positive": int(labels.sum()),
                "uncompressed_array_bytes": int(images.nbytes + masks.nbytes), "file_bytes": Path(cache_path).stat().st_size}


def run_original_preprocessing(cfg):
    prep = preparation_paths(cfg)
    if prep["mode"] == "full" and cfg.get("existing_cache"):
        raise ValueError("已配置 existing_cache；请用核查已有缓存单元，不覆盖文件。")
    fresh(prep["cache"], prep["cache_manifest"], prep["cache"].with_suffix(".partial.npz"))
    matching, adapter, details = checked_inputs(cfg)
    images, masks, labels, layers = [], [], [], []
    for i, row in matching.iterrows():
        patient = adapter.loc[row.mr]
        image, mask = cache_patient(row.reg_dir, patient["脑室体素最大层"])
        images.append(image)
        masks.append(mask)
        labels.append(int(float(patient.MOCA) <= 22))
        layers.append(four_slice_indices(patient["脑室体素最大层"])[0] + 1)
        if (i + 1) % 20 == 0 or i + 1 == len(matching):
            print(f"已生成 {i+1}/{len(matching)} 例四层输入", flush=True)
    # Recheck source bytes after reading all images, before publishing a cache.
    checked_inputs(cfg)
    prep["root"].mkdir(parents=True, exist_ok=True)
    partial = prep["cache"].with_suffix(".partial.npz")
    with partial.open("xb") as stream:
        np.savez_compressed(stream, images=np.stack(images), masks=np.stack(masks), labels=np.asarray(labels, dtype=np.int64),
                            mrs=matching.mr.to_numpy(dtype=str), max_layers=np.asarray(layers, dtype=np.int16))
    summary = validate_cache_file(partial, matching.mr.tolist(), labels, layers)
    partial.rename(prep["cache"])
    dump_json(prep["cache_manifest"], {"cache_sha256": digest(prep["cache"]), "audit_details_sha256": digest(prep["audit_details"]),
        "adapter_sha256": digest(paths(cfg)["adapter"]), "settings": preparation_settings(cfg), "summary": summary,
        "method": "original_four_slices_percentile_1_99.5_linear_images_nearest_masks_224",
        "patients": [{"mr": mr, "selected_slices_1based": details["patients"][mr]["selected_slices_1based"],
                       "annotation_scope": details["patients"][mr]["annotation_scope"]} for mr in matching.mr]})
    record(cfg, "preprocessing_" + prep["mode"], summary)
    return summary


def verify_existing_cache(cfg):
    prep = preparation_paths(cfg)
    fresh(prep["cache_manifest"])
    matching, adapter, details = checked_inputs(cfg)
    patients = adapter.loc[matching.mr]
    labels = (patients.MOCA.to_numpy(dtype=float) <= 22).astype(np.int64)
    layers = np.asarray([four_slice_indices(v)[0] + 1 for v in patients["脑室体素最大层"]], dtype=np.int16)
    summary = validate_cache_file(prep["cache"], matching.mr.tolist(), labels, layers)
    with np.load(prep["cache"], allow_pickle=False) as cache:
        images, masks = cache["images"], cache["masks"]
        for i, row in matching.iterrows():
            expected_images, expected_masks = cache_patient(row.reg_dir, layers[i])
            if not np.array_equal(images[i], expected_images) or not np.array_equal(masks[i], expected_masks):
                raise ValueError(f"已有缓存与当前原生影像重新计算的结果不同：{row.mr}。")
    checked_inputs(cfg)
    dump_json(prep["cache_manifest"], {"cache_sha256": digest(prep["cache"]), "audit_details_sha256": digest(prep["audit_details"]),
        "adapter_sha256": digest(paths(cfg)["adapter"]), "settings": preparation_settings(cfg), "summary": summary,
        "method": "existing_cache_compared_patient_by_patient_to_current_source",
        "patients": [{"mr": mr, "selected_slices_1based": details["patients"][mr]["selected_slices_1based"],
                       "annotation_scope": details["patients"][mr]["annotation_scope"]} for mr in matching.mr]})
    return summary



## 保留时间划分并保存患者清单

缓存准备完成后，`prepare_split_and_cohort` 按原规则处理 206 例：按检查日期和 MR 排序，以第 144 位患者所在日期分组，同日患者都进入开发队列。分界日期仍核对为 2026-06-10，开发五折仍使用原随机种子。

清单增加了原始 MoCA、取层记录、标注范围及空间核查标记。这些字段用于追溯；分类模型输入仍是原来的五个临床变量。样例模式不生成时间分组或训练队列。


In [ ]:
from __future__ import annotations

def verify_preparation(cfg):
    prep = preparation_paths(cfg)
    matching, adapter, details = checked_inputs(cfg)
    manifest = json.loads(prep["cache_manifest"].read_text(encoding="utf-8"))
    if (manifest["cache_sha256"] != digest(prep["cache"]) or manifest["audit_details_sha256"] != digest(prep["audit_details"])
            or manifest["adapter_sha256"] != digest(paths(cfg)["adapter"]) or manifest["settings"] != preparation_settings(cfg)):
        raise ValueError("缓存、临床表或核查记录与生成时的版本不同。")
    patients = adapter.loc[matching.mr]
    validate_cache_file(prep["cache"], matching.mr.tolist(), (patients.MOCA.to_numpy(dtype=float) <= 22).astype(int),
                        [four_slice_indices(v)[0] + 1 for v in patients["脑室体素最大层"]])
    return matching, adapter, details


def prepare_split_and_cohort(cfg):
    from sklearn.model_selection import StratifiedKFold
    if preparation_paths(cfg)["mode"] != "full":
        raise ValueError("样例缓存仅用于检查，不生成时间划分或训练队列。")
    p = paths(cfg)
    fresh(p["cohort"], p["temporal"] / "temporal_split.csv", p["data"] / "prepared_data_manifest.json")
    matching, adapter, details = verify_preparation(cfg)
    if len(matching) != 206 or int(cfg["expected_patients"]) != 206:
        raise ValueError("当前保留原206例时间划分，不根据缺失病例重新定义研究队列。")
    with np.load(p["cache"], allow_pickle=False) as cache:
        mrs, y, layers = cache["mrs"].astype(str), cache["labels"].astype(int), cache["max_layers"]
    dates = pd.to_datetime(adapter.loc[mrs, "检查时间"], format="%Y.%m.%d")
    ordered = pd.DataFrame({"mr": mrs, "date": dates.to_numpy()}).sort_values(["date", "mr"], kind="mergesort")
    boundary = str(ordered.iloc[143]["date"].date())
    if boundary != cfg["expected_boundary"]:
        raise ValueError(f"原第144例分界日期为{boundary}，与配置不同，请先核对患者。")
    configure_baseline(p["cache"], p["full_cv"], p["temporal"], p["adapter"], cfg.get("patients_root"))
    prepare_temporal_main()
    cohort = pd.read_csv(p["temporal"] / "temporal_split.csv", dtype={"mr": str})
    cohort["cache_index"] = cohort.mr.map({mr: i for i, mr in enumerate(mrs)})
    cohort["label"] = y[cohort.cache_index.to_numpy(dtype=int)]
    cohort["moca"] = adapter.loc[cohort.mr, "MOCA"].to_numpy(dtype=float)
    cohort["max_layer"] = layers[cohort.cache_index.to_numpy(dtype=int)]
    directories = matching.set_index("mr").reg_dir
    for sequence, filename in zip(("t1", "t2", "dwi", "manual_mask"), IMAGE_FILES + ["Untitled.nii.gz"]):
        cohort[sequence] = [str(Path(directories[mr]) / filename) for mr in cohort.mr]
    clinical_names = {"age": "年龄", "education": "受教育年限", "nihss": "NIHSS", "hcy": "同型半胱氨酸", "ua": "尿酸"}
    for key in CLINICAL:
        cohort[key] = pd.to_numeric(adapter.loc[cohort.mr, clinical_names[key]], errors="coerce").to_numpy()
    cohort["selected_slices_1based"] = [json.dumps(details["patients"][mr]["selected_slices_1based"]) for mr in cohort.mr]
    cohort["annotation_scope"] = [details["patients"][mr]["annotation_scope"] for mr in cohort.mr]
    cohort["physical_geometry_needs_review"] = [details["patients"][mr]["physical_geometry_needs_review"] for mr in cohort.mr]
    cohort["native_images_available"] = True
    cohort["development_fold"] = 0
    dev_index = cohort.index[cohort.split == "development"].to_numpy()
    dev_y = cohort.loc[dev_index, "label"].to_numpy()
    if np.bincount(dev_y, minlength=2).min() < 5:
        raise ValueError("开发队列类别数量不足五折。")
    for fold, (_, valid) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(dev_y, dev_y), 1):
        cohort.loc[dev_index[valid], "development_fold"] = fold
    cohort.to_csv(p["cohort"], index=False)
    dump_json(p["data"] / "cohort_summary.json", {"n": len(cohort), "split": cohort.split.value_counts().to_dict(), "boundary": boundary})
    prepared_files = [p["cohort"], p["adapter"], p["temporal"] / "temporal_split.csv", preparation_paths(cfg)["cache_manifest"]]
    dump_json(p["data"] / "prepared_data_manifest.json", {"artifacts": {str(f.relative_to(p["run"])): digest(f) for f in prepared_files},
        "cache_sha256": digest(p["cache"]), "notebook_definitions_sha256": source_snapshot(cfg), "boundary": boundary})
    record(cfg, "cohort")
    return cohort



## 整理可上传的分类数据包

这一步将四层缓存、临床工作表、患者划分、核查记录和七本 Notebook 放入 `upload_packages/`。服务器配置使用相对路径，患者清单中的本地影像路径会清空，并标明包内没有原生影像。

上传整个包后，可直接从 02 开始训练，再运行 03、04 和 06 的分类评价。配置读取时会核对文件校验值。导出的运行开关全部关闭，06 默认只比较 ResNet、临床和融合三组。影像组学与分割仍需另行准备原分辨率影像和局部标注流程。


In [ ]:
from __future__ import annotations

def export_classification_bundle(cfg):
    import shutil
    p = paths(cfg)
    if preparation_paths(cfg)["mode"] != "full":
        raise ValueError("样例模式不导出完整训练数据包。")
    verify_preparation(cfg)
    prepared = json.loads((p["data"] / "prepared_data_manifest.json").read_text(encoding="utf-8"))
    if prepared["cache_sha256"] != digest(p["cache"]) or prepared["notebook_definitions_sha256"] != source_snapshot(cfg):
        raise ValueError("数据准备完成后缓存或代码定义改变，请核对实验版本。")
    for name, checksum in prepared["artifacts"].items():
        if digest(p["run"] / name) != checksum:
            raise ValueError(f"准备好的数据文件发生变化：{name}")
    notebooks = sorted(cfg["_root"].glob("*.ipynb"))
    if len(notebooks) != 7:
        raise ValueError("请将7本 Notebook 保存在同一目录后导出。")
    destination = cfg["_root"] / "upload_packages" / (cfg["run_id"] + "_classification")
    fresh(destination)
    target_run = destination / "outputs" / cfg["run_id"]
    (target_run / "data").mkdir(parents=True)
    (target_run / "temporal").mkdir()
    filenames = ["data/clinical_adapter.xlsx", "data/clinical_adapter_meta.json", "data/cohort_summary.json",
        "data/patient_matching.csv", "data/matching_meta.json", "data/volume_audit.csv", "data/volume_audit_details.json",
        "data/cache_manifest.json", "data/excluded_missing_outcome.csv", "data/clinical_issues.csv",
        "temporal/temporal_split.csv", "temporal/baseline_comparison.csv", "temporal/temporal_split_summary.md"]
    artifacts = {}
    for name in filenames:
        source, target = p["run"] / name, target_run / name
        shutil.copy2(source, target)
        if digest(source) != digest(target):
            raise ValueError(f"复制校验失败：{name}")
        artifacts[str(target.relative_to(destination))] = digest(target)
    # Keep the local preparation manifest as provenance; its cohort contains native paths.
    source_preparation = target_run / "data" / "source_prepared_data_manifest.json"
    shutil.copy2(p["data"] / "prepared_data_manifest.json", source_preparation)
    artifacts[str(source_preparation.relative_to(destination))] = digest(source_preparation)
    target_cache = target_run / "data" / "resnet50_manual_cache.npz"
    shutil.copy2(p["cache"], target_cache)
    if digest(target_cache) != prepared["cache_sha256"]:
        raise ValueError("上传包中的缓存复制不完整。")
    artifacts[str(target_cache.relative_to(destination))] = digest(target_cache)
    cohort = pd.read_csv(p["cohort"], dtype={"mr": str})
    for column in ("t1", "t2", "dwi", "manual_mask"):
        cohort[column] = ""
    cohort["native_images_available"] = False
    cohort_path = target_run / "data" / "cohort.csv"
    cohort.to_csv(cohort_path, index=False)
    artifacts[str(cohort_path.relative_to(destination))] = digest(cohort_path)
    for notebook in notebooks:
        content = json.loads(notebook.read_text(encoding="utf-8"))
        for cell in content["cells"]:
            if cell["cell_type"] == "code":
                cell["outputs"], cell["execution_count"] = [], None
                if not cell.get("metadata", {}).get("embedded_definition"):
                    text = "".join(cell["source"])
                    text = re.sub(r"(?m)^((?:RUN_\w+|LOCK_ORIGINAL_MODELS|INSTALL_SPLITS|EVALUATE_DEVELOPMENT_SEGMENTATION|SHOW_CACHE_EXAMPLE|COMPARE_DEVELOPMENT_OOF)\s*=\s*)True\b", r"\g<1>False", text)
                    if notebook.name.startswith("06_"):
                        text = text.replace("INCLUDE_RADIOMICS = True", "INCLUDE_RADIOMICS = False")
                    cell["source"] = text.splitlines(keepends=True)
        dump_json(destination / notebook.name, content)
    config = {k: v for k, v in cfg.items() if not k.startswith("_")}
    config.update(excel_path=None, patients_root=None, existing_cache=None, outputs_root="outputs",
                  data_access="classification_cache_only")
    dump_json(destination / "config.json", config)
    if (cfg["_root"] / "requirements.txt").is_file():
        shutil.copy2(cfg["_root"] / "requirements.txt", destination / "requirements.txt")
    dump_json(destination / "classification_bundle.json", {"run_id": cfg["run_id"], "kind": "classification_cache_only",
        "artifacts": artifacts, "cache": str(target_cache.relative_to(destination)), "cohort": str(cohort_path.relative_to(destination)),
        "source_cohort_sha256": digest(p["cohort"]), "notebook_definitions_sha256": source_snapshot(cfg),
        "patients": len(cohort), "split": cohort.split.value_counts().to_dict(), "boundary": prepared["boundary"],
        "native_images_included": False, "note": "原生路径已清空；本包仅供分类、单序列消融和缓存空间中的解释。"})
    (destination / "使用说明.md").write_text("# 分类数据包\n\n上传整个目录并在服务器解压或打开。配置读取时会核对输入文件校验值。\n\n"
        "无需在服务器重跑00的数据准备，依次使用02训练、03单序列、04热图、06分类时间验证。"
        "所有运行开关默认关闭；06默认不纳入影像组学或自动掩模。\n\n"
        "此包含四层224×224缓存、临床工作表、患者分组和核查记录，不含原生三维影像。"
        "01和05需另行准备原分辨率数据与适合局部标注的流程。核查记录中的本地原始路径仅用于追溯。\n",
        encoding="utf-8")
    return {"directory": str(destination), "patients": len(cohort),
            "bytes": sum(f.stat().st_size for f in destination.rglob("*") if f.is_file()), "input_files": len(artifacts)}



## 1. 核对临床表的列名

先读出 Excel 的真实表头，再与 `config.json` 中的 `columns` 对照。尤其要确认 MR 编号、MoCA、检查时间和取层字段，后续患者匹配与结局定义都依赖这几列。修改配置后，重新运行前面的配置读取单元。


In [ ]:
if cfg.get("excel_path"):
    print(excel_headers(cfg))
else:
    print("先在config.json填写excel_path、patients_root和columns.mr等列映射。")


## 2. 整理临床工作表

生成 `clinical_adapter.xlsx`，把“影像号”、MoCA、取层和临床变量放到后续函数需要的位置。`clinical_issues.csv` 记录编号和日期问题；`excluded_missing_outcome.csv` 记录结局不可用的行。源表保留原样。

表中的同型半胱氨酸和尿酸列包含单位，配置已对应到实际列名。运行后检查前几行，确认数据没有错位。


In [ ]:
RUN_ADAPTER = False
if RUN_ADAPTER:
    adapter = prepare_adapter(cfg)
    display(adapter[["MR", "检查时间", "脑室体素最大层", "MOCA", "同型半胱氨酸", "尿酸", "_excel_row"]].head())


## 3. 匹配患者与影像目录

先查看选中患者是否找到了四个必需文件。全量模式会检查全部临床患者；样例模式只处理配置里的 `sample_mrs`。缺失记录保留在表中，避免因目录不全而悄悄改变研究队列。


In [ ]:
RUN_PATIENT_MATCHING = False
if RUN_PATIENT_MATCHING:
    matching = match_patients(cfg)
    selected = matching.loc[matching.selected]
    display(selected)
    display(selected.groupby(["matched", "reason"], dropna=False).size().rename("n"))


## 4. 检查选中患者的影像和标签

缺文件、方向不同、空间错位、标签值异常或取层越界都会写入 `reason`，处理后才能生成缓存。单位不明、DICOM 与 NIfTI 层间距不同等情况会写入 `warnings`，保持现有数组用于分类复现，同时留待物理尺度核查。

重点看未通过检查的病例，以及空间问题是否集中在某些序列。样例与全量结果分别保存。


In [ ]:
RUN_VOLUME_AUDIT = False
if RUN_VOLUME_AUDIT:
    audit = audit_volumes(cfg)
    display(audit)
    display(audit.groupby(["eligible", "reason"], dropna=False).size().rename("n"))
    display(audit.loc[audit.warnings.ne(""), ["mr", "warnings"]])


## 5. 生成四层缓存，或核对已有缓存

新生成和核对已有缓存二选一。两种方式都会核对患者、结局、取层和影像内容，并记录数据版本。

全量模式需要完整 206 例通过检查。样例模式只生成所选患者的缓存，文件保存在 `data/sample/`，用于检查处理结果。


In [ ]:
RUN_PREPROCESSING = False
RUN_VERIFY_EXISTING_CACHE = False
if RUN_PREPROCESSING and RUN_VERIFY_EXISTING_CACHE:
    raise ValueError("新生成和核对已有缓存请选择其中一种。")
if RUN_PREPROCESSING:
    display(run_original_preprocessing(cfg))
if RUN_VERIFY_EXISTING_CACHE:
    display(verify_existing_cache(cfg))
print("当前处理模式：", preparation_paths(cfg)["mode"])
print("当前缓存：", preparation_paths(cfg)["cache"])


## 6. 保存时间分组与开发折号

全量缓存完成后运行本步。时间划分保持原规则和分界日期，开发队列五折也保持原随机种子。运行后核对两组人数及阳性人数。

单例检查到上一节即可，本节会阻止把样例缓存当成完整队列使用。


In [ ]:
RUN_SPLIT = False

if RUN_SPLIT:
    cohort = prepare_split_and_cohort(cfg)
    display(cohort.groupby("split").agg(n=("mr", "size"), positives=("label", "sum")))


## 7. 查看患者清单和基线比较

全量处理完成后，核对患者编号、分组、标签、取层及标注范围。基线表用于比较开发和时间验证患者的临床特征。


In [ ]:
if preparation_paths(cfg)["mode"] == "full" and p["cohort"].exists():
    cohort = pd.read_csv(p["cohort"])
    display(cohort[["mr", "split", "label", "development_fold", "max_layer", "annotation_scope"]].head())
    if (p["temporal"] / "baseline_comparison.csv").exists():
        display(pd.read_csv(p["temporal"] / "baseline_comparison.csv"))


### 抽看一例缓存

下面展示缓存中第一位患者的三序列、四层影像，并叠加病灶轮廓。主要检查切片顺序、影像与轮廓是否对齐，以及缩放后有没有明显异常；这是对前面数值检查的补充。


In [ ]:
SHOW_CACHE_EXAMPLE = False
if SHOW_CACHE_EXAMPLE:
    import numpy as np
    import matplotlib.pyplot as plt
    with np.load(preparation_paths(cfg)["cache"], allow_pickle=False) as data:
        example, masks = data["images"][0], data["masks"][0]
        fig, axes = plt.subplots(3, 4, figsize=(12, 8))
        for c, sequence in enumerate(["T1-FLAIR", "T2", "DWI"]):
            for s in range(4):
                axes[c, s].imshow(example[s, c], cmap="gray", vmin=0, vmax=1)
                for label, color in enumerate(["cyan", "lime", "magenta"]):
                    if masks[s, label].any() and not masks[s, label].all():
                        axes[c, s].contour(masks[s, label], levels=[0.5], colors=[color])
                axes[c, s].set_title(f"{sequence}, slice {int(data['max_layers'][0]) + s}")
                axes[c, s].axis("off")
        plt.tight_layout()
        plt.show()


## 8. 导出分类上传包

全量核查、缓存和时间分组完成后运行。输出位于 `upload_packages/<run_id>_classification/`，上传这个目录即可。包内会保留空间核查记录，便于以后追溯，但不会复制原始 DICOM 或三维影像。


In [ ]:
RUN_EXPORT_CLASSIFICATION = False
if RUN_EXPORT_CLASSIFICATION:
    bundle = export_classification_bundle(cfg)
    display(bundle)
    print("请上传该目录中的全部文件；服务器从02开始。")


## 上传后从哪里继续

分类数据包内已有缓存、临床表和患者分组。服务器上从 02 开始训练，接着运行 03 的单序列实验、04 的热图，以及 06 的分类时间验证；无需重新读取原始 DICOM 或运行本页的数据生成步骤。

01 影像组学和 05 自动勾画仍需要原分辨率影像，并要按实际标注范围调整处理流程。当前已确认的样例只标注第 11～14 层，其他层不能直接作为无病灶标签。
